In [ ]:
%%sql -r dataframe_4
USE DATABASE database_name;
USE SCHEMA schema_name;

CREATE OR REPLACE VIEW TOTAL_ORDERS AS

WITH

/* 1) Sparse daily counts for delivered orders */
daily_raw AS (
  SELECT
      DATE(order_time) AS order_date,
      COUNT(*)         AS total_orders
  FROM silver_orders
  WHERE state = 'delivered'
  GROUP BY DATE(order_time)
),

/* 2) Bounds for calendar spine */
bounds AS (
  SELECT
      MIN(order_date) AS start_date,
      MAX(order_date) AS end_date
  FROM daily_raw
),

/* 3) Calendar spine: one row per day from start_date to end_date */
calendar AS (
  SELECT 
      DATEADD('day', seq4(), start_date) AS order_date
  FROM bounds,
       TABLE(GENERATOR(ROWCOUNT => 50000))  -- ensure this exceeds your date span
  WHERE DATEADD('day', seq4(), start_date) <= end_date
),

/* 4) Join sparse counts to the spine; fill gaps with zeros */
daily AS (
  SELECT
      c.order_date,
      COALESCE(d.total_orders, 0) AS total_orders
  FROM calendar c
  LEFT JOIN daily_raw d
    ON d.order_date = c.order_date
),

/* 5) YTD average through the previous day (resets each calendar year) */
with_ytd AS (
  SELECT
      d.*,
      AVG(d.total_orders) OVER (
          PARTITION BY YEAR(d.order_date)
          ORDER BY d.order_date
          ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
      ) AS ytd_avg_prev_day
  FROM daily d
)

SELECT
    w.order_date,
    w.total_orders,

    /* % change vs previous day (ratio). If yesterday=0 or NULL => 0.00 */
    CAST(
      COALESCE(
        (w.total_orders - LAG(w.total_orders) OVER (ORDER BY w.order_date))
        / NULLIF(LAG(w.total_orders) OVER (ORDER BY w.order_date), 0),
        0
      ) AS DECIMAL(7,2)
    ) AS pct_change_previous_day,

    /* % change vs 7 days ago (ratio). If 7-days-ago=0 or NULL => 0.00 */
    CAST(
      COALESCE(
        (w.total_orders - pw.total_orders)
        / NULLIF(pw.total_orders, 0),
        0
      ) AS DECIMAL(7,2)
    ) AS pct_change_7_days_ago,

    /* Today's orders vs YTD avg through yesterday (ratio). If YTD avg=0 or NULL => 0.0000 */
    CAST(
      COALESCE(
        (w.total_orders - w.ytd_avg_prev_day) / NULLIF(w.ytd_avg_prev_day, 0),
        0
      ) AS DECIMAL(10,4)
    ) AS pct_vs_ytd_average_prev_day

FROM with_ytd w
LEFT JOIN daily pw
  ON pw.order_date = DATEADD('day', -7, w.order_date)
ORDER BY w.order_date DESC;

In [ ]:
%%sql -r dataframe_3
USE DATABASE database_name;
USE SCHEMA schema_name;

CREATE OR REPLACE VIEW TOTAL_SALES AS

WITH

/* 1) Daily totals for delivered orders (sparse by nature) */
daily_raw AS (
  SELECT
      DATE(order_time) AS order_date,
      SUM(order_total) AS order_total
  FROM silver_orders
  WHERE state = 'delivered'
  GROUP BY DATE(order_time)
),

/* 2) Bounds to construct a calendar spine */
bounds AS (
  SELECT
      MIN(order_date) AS start_date,
      MAX(order_date) AS end_date
  FROM daily_raw
),

/* 3) Calendar spine from start_date to end_date (one row per day) */
calendar AS (
  SELECT 
      DATEADD('day', seq4(), start_date) AS order_date
  FROM bounds,
       TABLE(GENERATOR(ROWCOUNT => 50000)) -- make sure this exceeds your day span
  WHERE DATEADD('day', seq4(), start_date) <= end_date
),

/* 4) Join sparse sales to the calendar and fill gaps with zeros */
daily AS (
  SELECT
      c.order_date,
      COALESCE(d.order_total, 0) AS order_total
  FROM calendar c
  LEFT JOIN daily_raw d
    ON d.order_date = c.order_date
),

/* 5) YTD average through the previous day (resets each calendar year) */
with_ytd AS (
  SELECT
      d.*,
      AVG(d.order_total) OVER (
          PARTITION BY YEAR(d.order_date)
          ORDER BY d.order_date
          ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
      ) AS ytd_avg_prev_day
  FROM daily d
)

SELECT
    w.order_date,
    w.order_total,

    /* Day-over-day change (ratio). If yesterday = 0 or NULL => 0.00 */
    CAST(
      COALESCE(
        (w.order_total - LAG(w.order_total) OVER (ORDER BY w.order_date))
        / NULLIF(LAG(w.order_total) OVER (ORDER BY w.order_date), 0),
        0
      ) AS DECIMAL(10,2)
    ) AS pct_change_previous_day,

    /* Change vs 7 days ago (ratio). If 7-days-ago = 0 or NULL => 0.00 */
    CAST(
      COALESCE(
        (w.order_total - pw.order_total) / NULLIF(pw.order_total, 0),
        0
      ) AS DECIMAL(10,2)
    ) AS pct_change_7_days_ago,

    /* Today vs YTD average through yesterday (ratio). If YTD avg = 0 or NULL => 0.0000 */
    CAST(
      COALESCE(
        (w.order_total - w.ytd_avg_prev_day) / NULLIF(w.ytd_avg_prev_day, 0),
        0
      ) AS DECIMAL(10,4)
    ) AS pct_vs_ytd_average_prev_day

FROM with_ytd w
LEFT JOIN daily pw
  ON pw.order_date = DATEADD('day', -7, w.order_date)
ORDER BY w.order_date DESC;

In [ ]:
%%sql -r dataframe_1
USE DATABASE database_name;
USE SCHEMA schema_name;

CREATE OR REPLACE VIEW NEW_CUSTOMERS AS

WITH

/* 1) Identify real people by name + DOB */
real_people AS (
    SELECT
        first_name,
        last_name,
        date_of_birth,
        customer_id
    FROM silver_customer
    WHERE first_name IS NOT NULL
      AND last_name  IS NOT NULL
      AND date_of_birth IS NOT NULL
),

/* 2) Delivered orders only; join across ALL IDs for each person */
person_orders AS (
    SELECT
        rp.first_name,
        rp.last_name,
        rp.date_of_birth,
        o.order_time
    FROM real_people rp
    LEFT JOIN silver_orders o
      ON o.customer_id = rp.customer_id
     AND o.state = 'delivered'
),

/* 3) First delivered order date per real person (across all their IDs) */
first_order_per_person AS (
    SELECT
        first_name,
        last_name,
        date_of_birth,
        MIN(DATE(order_time)) AS first_order_date
    FROM person_orders
    GROUP BY first_name, last_name, date_of_birth
),

/* 4) Raw daily counts of new customers (exclude NULL first_order_date) */
raw_daily AS (
    SELECT
        first_order_date AS order_date,
        COUNT(*) AS new_customers
    FROM first_order_per_person
    WHERE first_order_date IS NOT NULL
    GROUP BY first_order_date
),

/* 5) Bounds for calendar spine */
bounds AS (
    SELECT
        MIN(order_date) AS start_date,
        MAX(order_date) AS end_date
    FROM raw_daily
),

/* 6) Calendar spine (fix ROWCOUNT literal) */
calendar AS (
    SELECT 
        DATEADD(day, seq4(), start_date) AS order_date
    FROM bounds,
         TABLE(GENERATOR(ROWCOUNT => 50000))
    WHERE DATEADD(day, seq4(), start_date) <= end_date
),

/* 7) Fill missing dates with zeros */
daily AS (
    SELECT
        c.order_date,
        COALESCE(r.new_customers, 0) AS new_customers
    FROM calendar c
    LEFT JOIN raw_daily r
      ON r.order_date = c.order_date
),

/* 8) YTD average through previous day, per calendar year */
with_ytd AS (
    SELECT
        d.*,
        AVG(d.new_customers) OVER (
            PARTITION BY YEAR(d.order_date)
            ORDER BY d.order_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ) AS ytd_avg_prev_day
    FROM daily d
)

/* 9) Final metrics */
SELECT
    w.order_date,
    w.new_customers,

    /* % change vs previous day (ratio; multiply by 100 for %) */
    CAST(COALESCE(
        CASE 
            WHEN LAG(w.new_customers) OVER (ORDER BY w.order_date) = 0 THEN NULL
            ELSE
                (w.new_customers - LAG(w.new_customers) OVER (ORDER BY w.order_date))
                / LAG(w.new_customers) OVER (ORDER BY w.order_date)
        END, 0
    ) AS DECIMAL(7,2)) AS pct_change_previous_day,

    /* % change vs 7 days ago */
    CAST(COALESCE(
        CASE 
            WHEN pw.new_customers = 0 THEN NULL
            ELSE
                (w.new_customers - pw.new_customers) / pw.new_customers
        END, 0
    ) AS DECIMAL(7,2)) AS pct_change_7_days_ago,

    /* vs YTD average through previous day */
    CAST(COALESCE(
        CASE 
            WHEN w.ytd_avg_prev_day IS NULL OR w.ytd_avg_prev_day = 0 THEN NULL
            ELSE
                (w.new_customers - w.ytd_avg_prev_day) / w.ytd_avg_prev_day
        END, 0
    ) AS DECIMAL(10,4)) AS pct_vs_ytd_average_prev_day

FROM with_ytd w
LEFT JOIN daily pw
  ON pw.order_date = DATEADD(day, -7, w.order_date)
ORDER BY w.order_date DESC;

In [ ]:
%%sql -r dataframe_5
USE DATABASE database_name;
USE SCHEMA schema_name;

CREATE OR REPLACE VIEW AVERAGE_PICK_UP_TIME AS

WITH

/* 1) Sparse daily averages from delivered orders only */
daily_raw AS (
  SELECT
      DATE(order_time) AS order_date,
      /* Average pickup time per order (minutes) for that day */
      AVG(DATEDIFF('minute', order_time, pickup_time)) AS average_pickup_time
  FROM silver_orders
  WHERE state = 'delivered'
    AND order_time IS NOT NULL
    AND pickup_time IS NOT NULL
  GROUP BY DATE(order_time)
),

/* 2) Bounds for the calendar spine */
bounds AS (
  SELECT
      MIN(order_date) AS start_date,
      MAX(order_date) AS end_date
  FROM daily_raw
),

/* 3) Calendar spine: one row per day between start_date and end_date */
calendar AS (
  SELECT 
      DATEADD('day', seq4(), start_date) AS order_date
  FROM bounds,
       TABLE(GENERATOR(ROWCOUNT => 50000))  -- ensure this exceeds your span in days
  WHERE DATEADD('day', seq4(), start_date) <= end_date
),

/* 4) Join sparse data to the spine. Fill missing days with 0 minutes. */
daily AS (
  SELECT
      c.order_date,
      COALESCE(d.average_pickup_time, 0) AS average_pickup_time
  FROM calendar c
  LEFT JOIN daily_raw d
    ON d.order_date = c.order_date
),

/* 5) YTD average through previous day (resets each calendar year) */
with_ytd AS (
  SELECT
      d.*,
      AVG(d.average_pickup_time) OVER (
          PARTITION BY YEAR(d.order_date)
          ORDER BY d.order_date
          ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
      ) AS ytd_avg_prev_day
  FROM daily d
)

SELECT
    w.order_date,
    w.average_pickup_time,

    /* % change vs previous day (ratio). Yesterday=0 or NULL -> 0.00 */
    CAST(
      COALESCE(
        (w.average_pickup_time - LAG(w.average_pickup_time) OVER (ORDER BY w.order_date))
        / NULLIF(LAG(w.average_pickup_time) OVER (ORDER BY w.order_date), 0),
        0
      ) AS DECIMAL(7,2)
    ) AS pct_change_previous_day,

    /* % change vs 7 days ago (ratio). 7-days-ago=0 or NULL -> 0.00 */
    CAST(
      COALESCE(
        (w.average_pickup_time - pw.average_pickup_time)
        / NULLIF(pw.average_pickup_time, 0),
        0
      ) AS DECIMAL(7,2)
    ) AS pct_change_7_days_ago,

    /* Today vs YTD avg through yesterday (ratio). YTD avg=0 or NULL -> 0.0000 */
    CAST(
      COALESCE(
        (w.average_pickup_time - w.ytd_avg_prev_day) / NULLIF(w.ytd_avg_prev_day, 0),
        0
      ) AS DECIMAL(10,4)
    ) AS pct_vs_ytd_average_prev_day

FROM with_ytd w
LEFT JOIN daily pw
  ON pw.order_date = DATEADD('day', -7, w.order_date)
ORDER BY w.order_date DESC;

In [ ]:
%%sql -r dataframe_6
USE DATABASE database_name;
USE SCHEMA schema_name;

CREATE OR REPLACE VIEW AVERAGE_DELIVERY_TIME AS

WITH 

daily AS (
  SELECT
      DATE(order_time) AS order_date,
      /* Average delivery time in minutes per day */
      AVG(DATEDIFF('minute', pickup_time, delivery_time)) AS average_delivery_time
  FROM silver_orders
  WHERE state = 'delivered'
    AND pickup_time IS NOT NULL
    AND delivery_time IS NOT NULL
  GROUP BY DATE(order_time)
),
with_ytd AS (
  SELECT
      d.*,
      /* YTD average EXCLUSIVE of the current day (up to yesterday) */
      AVG(d.average_delivery_time) OVER (
          PARTITION BY YEAR(d.order_date)
          ORDER BY d.order_date
          ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
      ) AS ytd_avg_prev_day
  FROM daily d
)
SELECT
    w.order_date,
    w.average_delivery_time,

    /* % change vs previous day (ratio) */
    CAST(COALESCE(
        CASE
            WHEN LAG(w.average_delivery_time) OVER (ORDER BY w.order_date) = 0
                THEN NULL
            ELSE
                (w.average_delivery_time 
                 - LAG(w.average_delivery_time) OVER (ORDER BY w.order_date))
                / LAG(w.average_delivery_time) OVER (ORDER BY w.order_date)
        END, 0) AS DECIMAL(7,2)) AS pct_change_previous_day,

    /* % change vs 7 days ago (ratio) */
    CAST(COALESCE(
        CASE
            WHEN pw.average_delivery_time IS NULL 
                 OR pw.average_delivery_time = 0
               THEN NULL
            ELSE
               (w.average_delivery_time - pw.average_delivery_time)
               / pw.average_delivery_time
        END, 0) AS DECIMAL(7,2)) AS pct_change_7_days_ago,

    /* Today's delivery time vs YTD average up to yesterday (decimal ratio) */
    CAST(COALESCE(
        CASE
            WHEN w.ytd_avg_prev_day IS NULL OR w.ytd_avg_prev_day = 0 THEN NULL
            ELSE
                (w.average_delivery_time - w.ytd_avg_prev_day)
                / w.ytd_avg_prev_day
        END, 0) AS DECIMAL(10,4)) AS pct_vs_ytd_average_prev_day

FROM with_ytd w
LEFT JOIN daily pw
  ON pw.order_date = DATEADD(day, -7, w.order_date)
ORDER BY w.order_date DESC;